# Dimensional — Surrogate Key Strategy

Compare three SK approaches on `silver.sellers` (run each **twice**):

1. `monotonically_increasing_id()` — unstable across runs
2. `row_number()` ordered by natural key — stable for fixed data
3. Deterministic hash of natural key — stable across runs

Documents choices for SCD Type 1, SCD Type 2, and why `dim_date` uses `date_key` instead.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.surrogate_keys as sk_module
importlib.reload(sk_module)

from src.dimensional.surrogate_keys import (
    SurrogateKeyConfig,
    assign_sk_monotonic,
    assign_sk_row_number,
    assign_sk_hash,
    run_surrogate_key_tests,
    SK_ANALYSIS,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SurrogateKeyConfig()
print("Loaded from:", sk_module.__file__)
print("Entities:", spark.table(config.source_table).count())

In [ ]:
source = spark.table(config.source_table)

mono1 = assign_sk_monotonic(source, config.natural_key)
mono2 = assign_sk_monotonic(source, config.natural_key)
row1 = assign_sk_row_number(source, config.natural_key)
row2 = assign_sk_row_number(source, config.natural_key)
hash1 = assign_sk_hash(source, config.natural_key)
hash2 = assign_sk_hash(source, config.natural_key)

display(hash1.orderBy(config.natural_key).limit(10))

In [ ]:
results = run_surrogate_key_tests(spark, config)
print("Strategy decisions:")
for k, v in SK_ANALYSIS.items():
    print(f"  {k}: {v}")
print()
for test in results["tests"]:
    print(test)

In [ ]:
import json

report = run_surrogate_key_tests(spark, config)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_surrogate_keys.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)